# 05 · Loughran-McDonald lexicon sentiment per section

**Reads** `stg_clean_10Ks` (dbt) and the LM master dictionary CSV · **Writes** `lm_lexicon`, `fact_lm_sentiment`

Buckets (per section, from word counts):

| Bucket | LM categories |
|---|---|
| Optimism | positive |
| Caution | uncertainty + constraining |
| Concern | negative + litigious |

`*_per_1k` is the count per 1,000 tokens; `*_share` is the bucket's share of all signal words. The lexicon CSV is not
in the repository (licence): download *Loughran-McDonald_MasterDictionary_1993-2024.csv* from
[Notre Dame SRAF](https://sraf.nd.edu/loughranmcdonald-master-dictionary/) into a Unity Catalog volume and set the
`lm_csv_path` widget.

In [ ]:
# Make the repository's Python package importable (Databricks Git folder: cwd is this notebook's folder)
import os, sys
ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

from trading_sentiment import table

In [ ]:
dbutils.widgets.text("lm_csv_path",
                     "/Volumes/workspace/sec_filings/filings/Loughran-McDonald_MasterDictionary_1993-2024.csv",
                     "LM dictionary CSV")
LM_CSV = dbutils.widgets.get("lm_csv_path")

## 1 · Lexicon table

In [ ]:
from pyspark.sql import functions as F

raw = spark.read.option("header", "true").csv(LM_CSV)

def flag(column):                       # LM columns hold 0 or the year a word was added
    return (F.col(column).cast("int") != 0).alias(f"is_{column.lower()}")

lexicon = (raw.select(F.lower("Word").alias("word"),
                      flag("Positive"), flag("Negative"), flag("Uncertainty"), flag("Litigious"), flag("Constraining"))
              .dropDuplicates(["word"]))
lexicon.write.format("delta").mode("overwrite").saveAsTable(table("lm_lexicon"))
print(lexicon.count(), "words")

## 2 · Tokenise each section and count lexicon hits

In [ ]:
sections = (spark.table(table("stg_clean_10Ks"))
    .selectExpr("cik", "company_name", "tickers", "filing_date", "accessionNumber", "market_cap", "sector",
                "stack(3, 'BUSINESS', bus_text, 'RISKS', risks_text, 'MDA', mda_text) as (section_name, section_text)")
    .where("section_text is not null and length(trim(section_text)) > 0"))

tokens = (sections
    .withColumn("token", F.explode(F.split(F.regexp_replace(F.lower("section_text"), r"[^a-z0-9 ]", " "), r"\s+")))
    .where("token != ''"))

lm = spark.table(table("lm_lexicon"))
counts = (tokens.join(lm, tokens.token == lm.word, "left")
    .groupBy("cik", "company_name", "tickers", "filing_date", "accessionNumber", "market_cap", "sector", "section_name")
    .agg(F.count("*").alias("total_tokens"),
         F.sum(F.when(F.col("is_positive"), 1).otherwise(0)).alias("pos_count"),
         F.sum(F.when(F.col("is_negative"), 1).otherwise(0)).alias("neg_count"),
         F.sum(F.when(F.col("is_uncertainty"), 1).otherwise(0)).alias("uncertainty_count"),
         F.sum(F.when(F.col("is_litigious"), 1).otherwise(0)).alias("litigious_count"),
         F.sum(F.when(F.col("is_constraining"), 1).otherwise(0)).alias("constraining_count")))

## 3 · Buckets, intensities and shares (NULL instead of division by zero)

In [ ]:
def safe_div(num, den):
    return F.when(den > 0, num / den)          # ANSI mode raises on x/0, so guard explicitly

scored = (counts
    .withColumn("optimism_raw", F.col("pos_count"))
    .withColumn("caution_raw", F.col("uncertainty_count") + F.col("constraining_count"))
    .withColumn("concern_raw", F.col("neg_count") + F.col("litigious_count"))
    .withColumn("token_k", F.col("total_tokens") / F.lit(1000.0))
    .withColumn("total_signal", F.col("optimism_raw") + F.col("caution_raw") + F.col("concern_raw"))
    .withColumn("optimism_per_1k", safe_div(F.col("optimism_raw"), F.col("token_k")))
    .withColumn("caution_per_1k", safe_div(F.col("caution_raw"), F.col("token_k")))
    .withColumn("concern_per_1k", safe_div(F.col("concern_raw"), F.col("token_k")))
    .withColumn("optimism_share", safe_div(F.col("optimism_raw"), F.col("total_signal")))
    .withColumn("caution_share", safe_div(F.col("caution_raw"), F.col("total_signal")))
    .withColumn("concern_share", safe_div(F.col("concern_raw"), F.col("total_signal"))))

# This write was commented out in the old notebook, although the dbt mart reads the table.
scored.write.format("delta").mode("overwrite").saveAsTable(table("fact_lm_sentiment"))
display(scored.orderBy(F.desc("market_cap"), F.desc("filing_date")).limit(20))